In [39]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [40]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
512,I enjoyed every moment of this beautiful film ...,positive
148,Principally it is the story of two men who wer...,positive
244,"If you want a complete waste of time, because ...",negative
469,This is the best picture about baseball since ...,positive
37,It was very refreshing to watch this beautiful...,positive


In [41]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Pragyan Kumar\AppData\Local\Temp\ipykernel_7448\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [42]:
df = normalize_text(df)
df.head()

,review,sentiment
512,enjoyed every moment beautiful film intrigued ...,positive
148,principally story two men part portuguese revo...,positive
244,want complete waste time pulling lint belly bu...,negative
469,best picture baseball since redford whacked na...,positive
37,refreshing watch beautiful movie director main...,positive


In [43]:
df['sentiment'].value_counts()

sentiment
negative    260
positive    240
Name: count, dtype: int64

In [44]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [45]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
512,enjoyed every moment beautiful film intrigued ...,1
148,principally story two men part portuguese revo...,1
244,want complete waste time pulling lint belly bu...,0
469,best picture baseball since redford whacked na...,1
37,refreshing watch beautiful movie director main...,1


In [46]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [47]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [49]:
!pip install dagshub


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import os
import re
import dagshub
import mlflow

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
ENV_PATH = os.path.join(PROJECT_ROOT, ".env")
PLACEHOLDER_TOKENS = {"replace_with_your_dagshub_token", "your_dagshub_token", ""}


def expand_env_vars(value):
    def replace_match(match):
        return os.environ.get(match.group(1), match.group(0))

    return re.sub(r"\$\{([^}]+)\}", replace_match, value)


def load_env_file(env_path):
    if not os.path.exists(env_path):
        raise FileNotFoundError(f".env file not found at: {env_path}")

    with open(env_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            value = expand_env_vars(value)
            os.environ[key] = value


def get_required_env(key):
    value = os.getenv(key)
    if not value:
        raise ValueError(f"Missing required environment variable: {key}")
    return value


def is_valid_secret(value):
    value = (value or "").strip()
    return value not in PLACEHOLDER_TOKENS and not value.lower().startswith("replace_with")


def use_local_mlflow(reason):
    local_tracking_uri = "file:" + os.path.join(PROJECT_ROOT, "mlruns")
    mlflow.set_tracking_uri(local_tracking_uri)
    print(f"Using local MLflow tracking at {local_tracking_uri}. {reason}")


load_env_file(ENV_PATH)

dagshub_token = os.getenv("DAGSHUB_USER_TOKEN", "").strip()

if get_required_env("USE_DAGSHUB").lower() == "true" and is_valid_secret(dagshub_token):
    os.environ["MLFLOW_TRACKING_USERNAME"] = get_required_env("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
    os.environ["DAGSHUB_USER_TOKEN"] = dagshub_token

    mlflow.set_tracking_uri(get_required_env("MLFLOW_TRACKING_URI"))
    dagshub.init(
        repo_owner=get_required_env("DAGSHUB_REPO_OWNER"),
        repo_name=get_required_env("DAGSHUB_REPO_NAME"),
        host=get_required_env("DAGSHUB_HOST"),
        mlflow=True
    )
else:
    if get_required_env("USE_DAGSHUB").lower() == "true":
        use_local_mlflow("DAGSHUB_USER_TOKEN is missing or still has the placeholder value.")
    else:
        use_local_mlflow("USE_DAGSHUB is false.")

mlflow.set_experiment("Logistic Regression Baseline")

Using local MLflow tracking at file:d:\code\Docker\MLOps-practice-projects-\Project_3\mlruns. DAGSHUB_USER_TOKEN is missing or still has the placeholder value.


<Experiment: artifact_location='file:D:\\code\\Docker\\MLOps-practice-projects-\\Project_3\\mlruns/682622947137695424', creation_time=1778903164069, experiment_id='682622947137695424', last_update_time=1778903164069, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [51]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-16 09:16:25,692 - INFO - Starting MLflow run...
2026-05-16 09:16:26,338 - INFO - Logging preprocessing parameters...
2026-05-16 09:16:26,348 - INFO - Initializing Logistic Regression model...
2026-05-16 09:16:26,350 - INFO - Fitting the model...
2026-05-16 09:16:26,395 - INFO - Model training complete.
2026-05-16 09:16:26,397 - INFO - Logging model parameters...
2026-05-16 09:16:26,401 - INFO - Making predictions...
2026-05-16 09:16:26,403 - INFO - Calculating evaluation metrics...
2026-05-16 09:16:26,416 - INFO - Logging evaluation metrics...
2026-05-16 09:16:26,431 - INFO - Saving and logging the model...
2026/05/16 09:16:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026-05-16 09:16:55,104 - INFO - Model training and logging completed in 28.77 seconds.
2026-05-16 09:16:55,105 - INFO - Accuracy: 0.664
2026-05-16 09:16:55,106 - INFO - Precision: 